# Segment-Level (classical) Analysis — store-backed

Classical Hopfield-solver segment metrics read from **`qtrk_store`** via `qtrk_view` (recomputed VIEW; fixed gamma-aware absolute threshold, never baked pkl metrics). Covers the multiplicity, scattering, resolution, hit-inefficiency and cone sweeps.

**Omitted vs the old notebook:** the spectral diagnostics (condition number kappa) and the tracker A/B (connected-components vs layered) — those need the matrix A and `get_tracks`, which the store does not persist. Re-derive separately if needed.

In [ ]:
# Setup
import sys, pathlib
sys.path.insert(0, '/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/_shared')
sys.path.insert(0, '/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Verify_new_results')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import qtrk_view as V

OUT = pathlib.Path('outputs/segment_analysis/store_backed'); OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size':12,'axes.grid':True,'grid.alpha':0.3,'lines.markersize':6,'lines.linewidth':1.8})
def _save(fig,stem):
    fig.savefig(OUT/f'{stem}.pdf', bbox_inches='tight'); fig.savefig(OUT/f'{stem}.png', dpi=150, bbox_inches='tight'); print('saved', stem)

# all CLASSICAL solves across studies (the segment-level characterisation set)
view = V.load_view()
C = view[view.solver=='classical'].copy()
print('classical solves:', len(C), '| studies:', sorted(C.study.unique()))
print('axes: T', sorted(C.n_trk.unique()), '| sigma_scatt', sorted(C.sigma_scatt.unique()),
      '| sigma_res', sorted(C.sigma_res.unique()), '| phi_max', sorted(C.phi_max.unique()),
      '| hit_ineff', sorted(C.hit_ineff.unique()))
def sweep(df, xcol, fixed):
    q=df.copy()
    for k,v in fixed.items(): q=q[np.isclose(q[k], v)] if q[k].dtype.kind=='f' else q[q[k]==v]
    g=q.groupby(xcol)
    out=pd.DataFrame({xcol:sorted(q[xcol].unique())}).set_index(xcol)
    for met in ['segment_efficiency','segment_false_rate','segment_purity']:
        out[met+'_m']=g[met].mean(); out[met+'_s']=g[met].apply(lambda x: x.std(ddof=1)/np.sqrt(len(x)) if len(x)>1 else 0.0)
    out['n']=g.size()
    return out.reset_index()
def plot_sweep(df, xcol, fixed, title, stem, logx=True, logy_far=True):
    s=sweep(df,xcol,fixed)
    fig,ax=plt.subplots(1,2,figsize=(13,4.5))
    if len(s):
        ax[0].errorbar(s[xcol],s['segment_efficiency_m'],yerr=s['segment_efficiency_s'],fmt='o-',capsize=4,color='#1b7837',label='efficiency')
        ax[0].errorbar(s[xcol],s['segment_purity_m'],yerr=s['segment_purity_s'],fmt='s--',capsize=4,color='#2166ac',label='purity')
        ax[1].errorbar(s[xcol],s['segment_false_rate_m'],yerr=s['segment_false_rate_s'],fmt='o-',capsize=4,color='#c51b7d')
    ax[0].set_ylim(0,1.05); ax[0].set_ylabel('efficiency / purity'); ax[0].legend(); ax[0].set_title(title+' — eff/purity')
    ax[1].set_ylabel('false rate = N_false_act/N_active'); ax[1].set_title(title+' — false rate')
    if logy_far: ax[1].set_yscale('log')
    for a in ax:
        a.set_xlabel(xcol)
        if logx: a.set_xscale('log')
    fig.tight_layout(); _save(fig,stem); plt.show()
    return s

In [ ]:
# Fig 1 — segment metrics vs track multiplicity (low-noise baseline)
plot_sweep(C, 'n_trk', dict(sigma_res=0.0, sigma_scatt=1e-4, phi_max=0.2, hit_ineff=0.0),
           'Multiplicity (clean, sigma_scatt=1e-4)', 'fig1_vs_T')

In [ ]:
# Fig 2 — scattering sweep (vs sigma_scatt), per fixed T
plot_sweep(C, 'sigma_scatt', dict(sigma_res=0.0, phi_max=0.2, hit_ineff=0.0, n_trk=100),
           'Scattering sweep (T=100, clean res)', 'fig2_vs_sigma_scatt', logx=True)

In [ ]:
# Fig 3 — resolution sweep (vs sigma_res), per fixed T
plot_sweep(C, 'sigma_res', dict(sigma_scatt=1e-4, phi_max=0.2, hit_ineff=0.0, n_trk=100),
           'Resolution sweep (T=100, sigma_scatt=1e-4)', 'fig3_vs_sigma_res', logx=False)

In [ ]:
# Fig 4 — hit-inefficiency sweep (vs hit_ineff / p_drop)
plot_sweep(C, 'hit_ineff', dict(sigma_res=0.0, sigma_scatt=1e-4, phi_max=0.2, n_trk=100),
           'Hit-inefficiency sweep (T=100)', 'fig4_vs_hit_ineff', logx=False)

In [ ]:
# Fig 5 — angular cone sweep (vs phi_max)
plot_sweep(C, 'phi_max', dict(sigma_res=0.0, sigma_scatt=1e-4, hit_ineff=0.0, n_trk=100),
           'Cone sweep (T=100, clean)', 'fig5_vs_phi_max', logx=True)

## Notes / provenance
- Source: `/data/bfys/gscriven/qtrk_store` via `qtrk_view`/`qtrk_pipeline.load_metrics` (classical solves, all studies). Metrics recomputed (gamma-aware absolute threshold; false rate = N_false_active/N_active).
- The fixed slices above pick a representative point on the other axes; adjust the `fixed=` dicts to slice differently. Coverage grows as Condor drains and `build_metrics.py` is re-run.
- **Not reproducible from the store:** condition-number / eigenspectrum (needs the matrix A, regenerate via `qtrk_pipeline.build_hamiltonian`) and tracker A/B (needs `get_tracks` on the solution).